Maison Intent Search: Proof of Concept

In this demonstration of Maison Intent Search, a Hugging Face Datasets dataset (published under the DBQ/namespace as “Louis.Vuitton.Product.prices.<Market>” - e.g., Singapore, France, UK) was utilized to emulate an internal Louis Vuitton product catalog dataset, given that none directly from Louis Vuitton could be accessed for third-party use. These datasets contain web-scraped Louis Vuitton product listings and prices for a specific country site at the time of capture. Each row represents a single Louis Vuitton product listed on the brand’s site for that specific market, typically including details such as:

Product name (title)
Category hierarchy/navigation path
E.g., Women → Handbags → … etc

Product URL and image URL
Price and currency
(depending on the market dataset) a date/capture timestamp

To provide a Proof of Concept for Maison Intent Search, product name + category path (+ URL as optional context) was used to build a searchable “document” per Louis Vuitton item, then match client intent queries to specific products via TF-IDF + cosine similarity.


Step 1: Downloads the dataset and converts it to a pandas DataFrame.

In [1]:
!pip -q install datasets

from datasets import load_dataset
import pandas as pd

ds = load_dataset("DBQ/Louis.Vuitton.Product.prices.Singapore")

lv = ds["train"].to_pandas()

lv.shape, lv.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-e177ee175f02ea(…):   0%|          | 0.00/748k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6576 [00:00<?, ? examples/s]

((6576, 17),
     website_name competence_date country_code currency_code          brand  \
 0  Louis Vuitton      2023-11-17          SGP           SGD  LOUIS VUITTON   
 1  Louis Vuitton      2023-11-17          SGP           SGD  LOUIS VUITTON   
 2  Louis Vuitton      2023-11-17          SGP           SGD  LOUIS VUITTON   
 3  Louis Vuitton      2023-11-17          SGP           SGD  LOUIS VUITTON   
 4  Louis Vuitton      2023-11-17          SGP           SGD  LOUIS VUITTON   
 
   category1_code category2_code                 category3_code  \
 0            NEW        FOR MEN  PRE COLLECTION FALL WINTER 23   
 1            NEW  READY TO WEAR             SEASONAL SELECTION   
 2            MEN  READY TO WEAR             SEASONAL SELECTION   
 3          WOMEN  READY TO WEAR                        DRESSES   
 4          WOMEN  READY TO WEAR        PREFALL 2023 COLLECTION   
 
             product_code                                     title  \
 0  NVPROD4330232V/1ABJ0A           

In [2]:
lv.columns

Index(['website_name', 'competence_date', 'country_code', 'currency_code',
       'brand', 'category1_code', 'category2_code', 'category3_code',
       'product_code', 'title', 'itemurl', 'imageurl', 'full_price', 'price',
       'full_price_eur', 'price_eur', 'flg_discount'],
      dtype='object')

Step 2: Build a product “document” column that TF-IDF learns from
Here, likely text columns are automatically selected (if they exist) and are concatenated.

In [3]:
def safe_str(x):
    return "" if pd.isna(x) else str(x)

text_fields = []

candidates = ["product_name", "name", "brand", "gender", "category", "category_1", "category_2",
              "category_3", "category_4", "url"]
text_fields = [c for c in candidates if c in lv.columns]

lv["doc"] = lv.apply(lambda r: " ".join(safe_str(r[c]) for c in text_fields), axis=1)

lv[["doc"]].head()

,doc
0,LOUIS VUITTON
1,LOUIS VUITTON
2,LOUIS VUITTON
3,LOUIS VUITTON
4,LOUIS VUITTON


Step 3: Create a single text field per product (“document”, in this case) so TF-IDF can be applied consistently.

In [4]:
import numpy as np

def safe_str(x):
    return "" if pd.isna(x) else str(x)

candidate_cols = [
    "product_name", "name", "title",
    "brand",
    "gender",
    "category", "category_1", "category_2", "category_3", "category_4",
    "category_path", "breadcrumb",
    "url"
]
text_cols = [c for c in candidate_cols if c in lv.columns]

print("Using text columns:", text_cols)

lv["doc_raw"] = lv.apply(lambda r: " ".join(safe_str(r[c]) for c in text_cols), axis=1)

lv = lv[lv["doc_raw"].str.strip().astype(bool)].reset_index(drop=True)

lv[["doc_raw"]].head(3)

Using text columns: ['title', 'brand']


,doc_raw
0,Convertible Ripstop Shirt LOUIS VUITTON
1,Monogram French Terry Zip-Through Hoodie LOUIS...
2,Workwear Denim Jacket LOUIS VUITTON


Step 4: Standard text normalization so TF-IDF focuses on meaningful terms, as opposed to unevenly weighing noise.


In [5]:
import re

def clean_text(s: str) -> str:
    s = safe_str(s).lower()
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

lv["doc"] = lv["doc_raw"].apply(clean_text)
lv[["doc_raw", "doc"]].head(3)

,doc_raw,doc
0,Convertible Ripstop Shirt LOUIS VUITTON,convertible ripstop shirt louis vuitton
1,Monogram French Terry Zip-Through Hoodie LOUIS...,monogram french terry zip through hoodie louis...
2,Workwear Denim Jacket LOUIS VUITTON,workwear denim jacket louis vuitton


Step 5: Convert each product document into a TF-IDF vector. Here, rows = Products, columns = Vocabulary Terms, and values = Importance Weights.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90
)

X = tfidf.fit_transform(lv["doc"])
print("TF-IDF matrix shape:", X.shape)

TF-IDF matrix shape: (6576, 4170)


Step 6: Adds Louis Vuitton, brand-relevant synonyms so the system better understands phrases of high-level intent.

In [7]:
SYNONYMS = {
    "quiet luxury": "understated minimalist subtle",
    "work": "office professional tailored",
    "travel": "lightweight durable compact",
    "everyday": "daily versatile practical",
    "wedding": "occasion formal evening",
    "gift": "present timeless classic"
}

def expand_query(q: str) -> str:
    q = clean_text(q)
    for k, v in SYNONYMS.items():
        if k in q:
            q += " " + v
    return q

Step 7:
1. Transforms the query into TF-IDF space
2. Computes cosine similarity to every product
3. Returns top matches + “why this” terms to ensure interpretability


In [8]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

feature_names = np.array(tfidf.get_feature_names_out())

def explain_match(query_vec, doc_vec, topn=8):
    """Return top overlapping TF-IDF terms between query and a product doc."""
    q = query_vec.toarray().ravel()
    d = doc_vec.toarray().ravel()
    overlap = q * d
    top_idx = overlap.argsort()[::-1][:topn]
    terms = [feature_names[i] for i in top_idx if overlap[i] > 0]
    return ", ".join(terms[:topn])

def recommend(query: str, top_k: int = 5) -> pd.DataFrame:
    q = expand_query(query)
    q_vec = tfidf.transform([q])

    sims = cosine_similarity(q_vec, X).ravel()
    top_idx = np.argsort(sims)[::-1][:top_k]

    display_cols = []
    for c in ["product_name", "name", "title"]:
        if c in lv.columns:
            display_cols.append(c)
            break
    for c in ["price", "currency", "url", "category_path", "category", "category_1", "category_2"]:
        if c in lv.columns:
            display_cols.append(c)

    results = lv.loc[top_idx, display_cols].copy()
    results["similarity"] = sims[top_idx]

    results["why_this"] = [
        explain_match(q_vec, X[i], topn=8) for i in top_idx
    ]

    return results.reset_index(drop=True)

recommend("minimalist black bag for work that fits a laptop", top_k=5)

,title,price,similarity,why_this
0,8 Ink Cartridges Black,16.0,0.284578,black
1,Damier Black Ring,555.0,0.262863,black
2,Silver Lockit Beads Bracelet Black Titanium a...,1060.0,0.254969,black
3,LV Initiales 40mm Matte Black Belt,740.0,0.253075,black
4,Black Monogram Andrews Golf Kit,1270.0,0.251074,black


Maison Intent Search Demo Queries:

In [9]:
demo_queries = [
    "quiet luxury everyday bag",
    "travel luggage lightweight durable",
    "gift classic timeless fragrance",
    "minimalist work bag understated",
    "evening formal accessory"
]

for q in demo_queries:
    print("\nQUERY:", q)
    display(recommend(q, top_k=3))


QUERY: quiet luxury everyday bag


,title,price,similarity,why_this
0,Card Holder Daily,395.0,0.378935,daily
1,Daily Pouch,1320.0,0.338245,daily
2,Daily Pouch,1050.0,0.338245,daily



QUERY: travel luggage lightweight durable


,title,price,similarity,why_this
0,3D Monogram Lightweight Parka,6450.0,0.589252,lightweight
1,Embroidered Lightweight Bomber,4900.0,0.550684,lightweight
2,Twist Compact Wallet,1420.0,0.182074,compact



QUERY: gift classic timeless fragrance


,title,price,similarity,why_this
0,Classic Shirt,2960.0,0.509933,classic
1,Monogram Classic Tie,355.0,0.492684,classic
2,Blurry Apples Single-Breasted Classic Jacket,11100.0,0.441599,classic



QUERY: minimalist work bag understated


,title,price,similarity,why_this
0,Tailored Casual Pants,1760.0,0.283296,tailored
1,Fluid Wool Tailored Pants,2960.0,0.273665,tailored
2,Bag Holder,560.0,0.244461,bag



QUERY: evening formal accessory


,title,price,similarity,why_this
0,Evening Blouson,5500.0,0.706499,evening
1,Evening Cigarette Pants,1950.0,0.556757,evening
2,All About MNG Evening Tie,560.0,0.497004,evening


Exploratory Step 8: Groups products into coarse themes, mirroring the categories found online (including travel, small leather goods, accessories, etc), which become “intent shelves” within the Maison Intent Search process on the e-commerce site.


In [10]:
from sklearn.cluster import MiniBatchKMeans

K = 8
km = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=2048)
lv["cluster"] = km.fit_predict(X)

lv["cluster"].value_counts().sort_index()

,count
cluster,
0,4180
1,288
2,213
3,605
4,649
5,176
6,221
7,244


In [11]:
import numpy as np

terms = tfidf.get_feature_names_out()
centroids = km.cluster_centers_

for k in range(K):
    top = np.argsort(centroids[k])[::-1][:12]
    print(f"\nCluster {k} top terms:")
    print(", ".join(terms[i] for i in top))


Cluster 0 top terms:
lv, mm louis, bandoulière, bandoulière louis, blossom, pm, mm, gold, pm louis, keepall, holder, keepall bandoulière

Cluster 1 top terms:
shirt louis, shirt, sleeved, cotton, short sleeved, short, cotton shirt, monogram, sleeved shirt, sleeved cotton, long sleeved, long

Cluster 2 top terms:
sneaker, sneaker louis, trainer sneaker, lv trainer, trainer, lv, run, lv archlight, run away, away sneaker, archlight, time sneaker

Cluster 3 top terms:
lv, bracelet, bracelet louis, belt louis, belt, mm, mm reversible, reversible, reversible belt, lv volt, volt, necklace

Cluster 4 top terms:
monogram, jacket louis, jacket, denim, jacquard, monogram jacquard, denim jacket, shawl louis, shawl, tile, monogram flower, flower

Cluster 5 top terms:
pants louis, pants, cigarette pants, cigarette, denim pants, denim, monogram, wool, jogging pants, jogging, tailored, cargo pants

Cluster 6 top terms:
wallet, wallet louis, multiple wallet, multiple, lv yk, yk, wearable, wearable wal

This initial prototype of Maison Intent Search uses a publicly available Hugging Face dataset of Louis Vuitton product listings and pricings, specific to certain market’s catalog snapshots. For each product, I created one text “document” by concatenating product name and category hierarchy fields. Text normalization was performed, including lowercasing and removing numbers/punctuation. I then fit a TF-IDF vectorizer to represent each product document as a weighted term vector. Depending on a client’s natural-language query, I transformed the query into the same TF-IDF space, computed cosine similarity to all products, and returned the top-ranked resulting products. In an effort to keep the system aligned with the Louis Vuitton brand and for the sake of interpretability, I expanded the results to output a “why this” explanation detailing the highest-overlap TF-IDF terms between the query and each recommended product. This step adds a layer of attentiveness and luxury to the e-commerce experience, redefining the act of online shopping from a transactional, cold process to a reimagined, technological extension of interacting with a Sales Associate.
